# German punctuality pipeline (DB Mobilithek data)

Same logic as `Swiss_pipeline.ipynb` (itself a single-source version of `Chuuchuu_pipeline.ipynb`), on the **raw DB punctuality data from Mobilithek**
(`DB Mobilithek data/DB_punctuality_2025.tar`: one parquet per service day). One notebook run = one month; `run_german_pipeline.py` runs the 12 months with papermill.

What differs from the Swiss pipeline, and why:

| Swiss pipeline | This notebook |
|---|---|
| one row per stop, `FAELLT_AUS_TF`, `*_PROGNOSE_STATUS` | the file is a **forecast history**: one row per (trip, stop, arrival/departure) **and per update**. `is_final` marks the last version, only those rows are kept |
| no status of the delay | no measured / forecast status either: **the final snapshot is taken as the actual delay (same as the Swiss pipeline)**. Many final snapshots predate the stop: reported in the sanity checks, not corrected |
| local Swiss time | UTC in the file, converted to `Europe/Berlin` |
| one row per stop, arrival and departure on the same row | separate arrival and departure rows, merged here into one stop row |
| terminus = last stop sorted by planned time | terminus = highest `stop_sequence` |
| all stops Swiss | the feed also holds stops abroad (cross-border trains): **punctuality units and the all-stops sheet only use the German stops** (closing stop = last German stop); `journey_type` uses all the stops |
| operator in the feed (`BETREIBER_ABK`) | operator is a **UIC company code** (`80` DB, `81` OEBB...), translated with `OPERATOR_NAMES` |
| all products | **long distance only** (`LD_CATEGORIES`), as asked |
| 3, 5, 15 min all-stops thresholds | 3, 5, **6**, 15 min, plus **6 min punctuality with cancelled stops left out** (to compare with the official DB figure) |
| - | days without real-time data (September, a few other days) are detected and **excluded** |

Steps: 0 load - 1 stop-level preparation - 2 terminus identification - 3 operator and service flags - 4 cancellation - 5 punctuality units & summaries - 6 export - 7 sanity checks.

In [ ]:
# Parameters
data_selection = "2025_01"      # month to process, e.g. "2025_01"
export_enriched = "n"           # "y" writes the per-stop enriched parquet next to the raw data (processed_data/)
export_summary = "y"            # "y" writes summary_stats/german_summary_<month>.xlsx

In [ ]:
import io
import os
import tarfile
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

from chuuchuu_paths import DB_MOBILITHEK_DIR, DB_MOBILITHEK_TAR

# --- Methodology choices, all in one place -------------------------------------------------------
# Long distance only. Categories seen on long-distance trains in the feed (THA kept in case it appears).
LD_CATEGORIES = ["ICE", "IC", "EC", "ECE", "NJ", "EN", "RJ", "RJX", "TGV", "FLX", "ES", "THA"]
HIGHSPEED_CATEGORIES = {"ICE", "TGV", "RJX", "THA"}
NIGHT_CATEGORIES = {"EN", "NJ", "ES"}
# The final snapshot is taken as the actual delay (same choice as the Swiss pipeline) : no measured / forecast status in the feed.
# A day whose final rows mostly have no update timestamp has no real-time data at all (schedule only) : excluded.
MAX_NULL_UPDATE_SHARE = 0.5
DELAY_THRESHOLDS_MIN = [3, 5, 6, 10, 15, 30, 60]     # terminus summary (same as Chuuchuu / Swiss)
ALL_STOPS_THRESHOLDS_MIN = [3, 5, 6, 15]             # all-stops sheet : one set of columns per threshold
COUNTRY = "Germany"
TIMEZONE = "Europe/Berlin"
# The operator field is a UIC company code (RICS), mostly the UIC country code of the incumbent. Check before publishing.
OPERATOR_NAMES = {
    "80": "DB", "81": "OEBB", "82": "CFL", "83": "Trenitalia", "84": "NS", "85": "SBB", "86": "DSB", "87": "SNCF", "88": "SNCB",
    "51": "PKP", "54": "CD", "55": "MAV", "56": "ZSSK", "74": "SJ", "78": "HZ", "79": "SZ", "22": "UZ",
}


def report_memory(label):
    gb = data.memory_usage(deep=True).sum() / 1e9
    print(f"[memory] {label}: {gb:.2f} GB (data)")

# Step 0 - Raw data loading

The tar is read in place, one day at a time (no extraction). For each day of the month:
- keep only the `is_final` rows (the last version of every stop row; checked: exactly one per trip / stop sequence / arrival-or-departure),
- **exclude the day if more than `MAX_NULL_UPDATE_SHARE` of its final rows have no update timestamp** (no real-time data: September 2025 and a few days in January / April),
- keep the long-distance categories, drop `is_betriebshalt` rows (operational stops, no passengers).

In [ ]:
COLUMNS = ["trip_id", "stop_sequence", "is_arrival", "time_schedule", "time_real", "delay", "is_cancelled", "update_timestamp",
           "stop_id", "operator", "category", "number", "line", "is_betriebshalt", "is_final"]
year_str, month_str = data_selection.split("_")
day_prefix = f"{year_str}-{month_str}-"

frames, day_log = [], []
with tarfile.open(DB_MOBILITHEK_TAR) as tar:
    members = sorted(name for name in tar.getnames() if name.startswith(day_prefix) and name.endswith(".parquet"))
    if not members:
        raise FileNotFoundError(f"No day of {data_selection} in {DB_MOBILITHEK_TAR}")
    for member in members:
        day = member[:10]
        raw = pq.read_table(io.BytesIO(tar.extractfile(member).read()), columns=COLUMNS).to_pandas()
        final = raw[raw["is_final"]]
        null_update_share = final["update_timestamp"].isna().mean()
        usable_day = null_update_share <= MAX_NULL_UPDATE_SHARE
        kept = final[final["category"].isin(LD_CATEGORIES) & ~final["is_betriebshalt"]]
        day_log.append({"day": day, "raw_rows": len(raw), "final_rows": len(final), "null_update_share": round(float(null_update_share), 4),
                        "ld_final_rows": len(kept), "status": "kept" if usable_day else "excluded (no real-time data)"})
        if usable_day:
            frames.append(kept.assign(day=day))
        del raw, final

day_log = pd.DataFrame(day_log)
excluded_days = day_log.loc[day_log["status"] != "kept", "day"].tolist()
print(f"{len(members)} days in {data_selection}, {len(excluded_days)} excluded: {excluded_days}")
if not frames:
    raise RuntimeError(f"{data_selection}: every day is excluded, nothing to compute")
data = pd.concat(frames, ignore_index=True)
del frames
data = data.drop(columns=["is_final", "is_betriebshalt"])
row_counts = {"initial": int(day_log.loc[day_log["status"] == "kept", "final_rows"].sum()), "after_ld_filter": len(data)}
print(f"{len(data):,} long-distance final rows kept; categories {data['category'].value_counts().to_dict()}")
day_log

# Step 1 - Stop-level preparation

- **Local time**: UTC in the file, converted to `Europe/Berlin` (service day = the day of the file, a train can run past midnight).
- **One row per stop**: arrival and departure rows are merged on `(day, trip_id, stop_sequence)` into `plannedArrival`, `arrival`, `arrivalDelay`, `arrivalCancelled`, same for departure. About 10% of stops have no arrival row (first stops).
- **Delay in seconds** = the `delay` column (checked: real - schedule). It is always set, there is no measured / forecast status: the **final snapshot is taken as the actual delay**, like the Swiss pipeline.
- **Cancelled stop** = the arrival row's flag, or the departure row's when there is no arrival row.
- **Country** from the UIC prefix of `stop_id` (same table as the other pipelines). The feed also holds stops abroad on cross-border trains.

In [ ]:
for col in ["time_schedule", "time_real", "update_timestamp"]:
    data[col] = data[col].dt.tz_convert(TIMEZONE).dt.tz_localize(None)

key = ["day", "trip_id", "stop_sequence"]
static = data.drop_duplicates(key)[key + ["stop_id", "operator", "category", "number", "line"]]
arrivals = data[data["is_arrival"]].rename(columns={
    "time_schedule": "plannedArrival", "time_real": "arrival", "delay": "arrivalDelay",
    "is_cancelled": "arrivalCancelled", "update_timestamp": "arrival_update"})
departures = data[~data["is_arrival"]].rename(columns={
    "time_schedule": "plannedDeparture", "time_real": "departure", "delay": "departureDelay",
    "is_cancelled": "departureCancelled", "update_timestamp": "departure_update"})
assert not arrivals.duplicated(key).any() and not departures.duplicated(key).any(), "more than one final row for a trip / stop / arrival-or-departure"
n_input_rows = len(data)
data = (static
        .merge(arrivals[key + ["plannedArrival", "arrival", "arrivalDelay", "arrivalCancelled", "arrival_update"]], on=key, how="left")
        .merge(departures[key + ["plannedDeparture", "departure", "departureDelay", "departureCancelled", "departure_update"]], on=key, how="left"))
del arrivals, departures, static
print(f"{n_input_rows:,} arrival / departure rows -> {len(data):,} stop rows")
row_counts["stop_rows"] = len(data)

for col in ["arrivalCancelled", "departureCancelled"]:
    data[col] = data[col].astype("boolean")
data["cancelled"] = data["arrivalCancelled"].fillna(data["departureCancelled"]).fillna(False).astype(bool)
data["date"] = pd.to_datetime(data["day"])
data["routeType"] = data["category"].astype(str).str.strip().str.upper()
data["operator_code"] = data["operator"].astype(str).str.strip()

# country
uic_codes = pd.read_csv("sup_data/UIC_country_codes.csv", sep=";")
uic_codes["Numerical code"] = uic_codes["Numerical code"].astype(str)
stop_id = data["stop_id"].astype(str)
data["country"] = stop_id.str[:2].map(uic_codes.drop_duplicates("Numerical code").set_index("Numerical code")["Country"])
data.loc[stop_id.str.len() != 7, "country"] = "ERROR_INVALID_STOP_ID"
data["country"] = data["country"].fillna("UNKNOWN_COUNTRY")
print(data["country"].value_counts(dropna=False).head(12).to_dict())
print(f"arrivalDelay set on {data['arrivalDelay'].notna().mean() * 100:.1f}% of rows, departureDelay on {data['departureDelay'].notna().mean() * 100:.1f}%")

for col in ["routeType", "operator_code", "country", "line", "number", "category"]:
    data[col] = data[col].astype("category")
data = data.drop(columns=["operator"])
report_memory("after Step 1")

# Step 2 - Terminus identification

A journey is a service day + `trip_id` (`journey_id`). Stops are ordered by `stop_sequence`; first = `depart`, last = `terminus`, the rest `intermediate`. A journey with a single stop stays `unknown`.
These labels cover **all** the stops of the journey (German or not). The punctuality unit of Step 5 only uses the German stops: its closing stop is the last German stop.

`journey_type` is `international` when the journey calls at a stop in a valid country other than Germany, `domestic` otherwise.

In [ ]:
data["sort_time"] = data["plannedArrival"].fillna(data["plannedDeparture"])
data["journey_id"] = data["day"] + "|" + data["trip_id"].astype(str)
data["jcode"] = pd.factorize(data["journey_id"])[0]
data = data.sort_values(["jcode", "stop_sequence"], kind="stable").reset_index(drop=True)

j = data["jcode"].to_numpy()
is_first = np.r_[True, j[1:] != j[:-1]]
is_last = np.r_[j[1:] != j[:-1], True]
n_stops_in_journey = data.groupby("jcode")["stop_sequence"].transform("size")
usable = n_stops_in_journey > 1
data["depart_terminus"] = np.select([usable & is_first, usable & is_last, usable], ["depart", "terminus", "intermediate"], default="unknown")
print(data["depart_terminus"].value_counts().to_dict())
print(f"{(data['depart_terminus'] == 'unknown').mean() * 100:.2f}% of rows have unknown depart_terminus")

is_foreign_stop = ~data["country"].isin([COUNTRY, "ERROR_INVALID_STOP_ID", "UNKNOWN_COUNTRY"])
data["journey_type"] = np.where(is_foreign_stop.groupby(data["jcode"]).transform("any"), "international", "domestic")
print(data.drop_duplicates("jcode")["journey_type"].value_counts().to_dict())

# Step 3 - Operator and service flags

(Chuuchuu Steps 4 and 6.) The feed gives the operator of each trip as a UIC company code, translated with `OPERATOR_NAMES`; **unmapped codes are kept as `OTHER <code>`** and listed below so the table can be completed. One operator per trip (checked).
`operator_for_punctuality` = `normalized_operator`: every operator is scored on the German network, there is no incumbent rule here.

`HSR_service`, `night_train` follow the Chuuchuu definitions; `LD` is `Yes` for every row (long-distance scope).

In [ ]:
code_as_str = data["operator_code"].astype(str)
data["normalized_operator"] = code_as_str.map(OPERATOR_NAMES)
flixtrain = code_as_str.str.upper().str.startswith("FLX")
data.loc[flixtrain, "normalized_operator"] = "FLIXTRAIN"
data["normalized_operator"] = data["normalized_operator"].fillna("OTHER " + code_as_str)
data["operator_for_punctuality"] = data["normalized_operator"]

data["HSR_service"] = np.where(data["routeType"].isin(HIGHSPEED_CATEGORIES), "Yes", "No")
data["night_train"] = np.where(data["routeType"].isin(NIGHT_CATEGORIES), "Yes", "No")
data["LD"] = "Yes"

unmapped = data.loc[data["normalized_operator"].str.startswith("OTHER "), "operator_code"].astype(str).value_counts()
print(f"{len(unmapped)} unmapped operator codes ({unmapped.sum() / len(data) * 100:.2f}% of rows): {unmapped.head(15).to_dict()}")
print(data.groupby("normalized_operator", observed=True).size().sort_values(ascending=False).head(15).to_dict())
print(data.groupby(["routeType", "normalized_operator"], observed=True).size().unstack(fill_value=0).sum(axis=1).sort_values(ascending=False).to_dict())
report_memory("after Step 3")

# Step 4 - Cancellation status

`is_cancelled` is set on every arrival / departure row, so `arrivalCancelled_resolved` / `departureCancelled_resolved` are that flag in the Chuuchuu `'t'` / `'f'` coding, with no null inference (a missing arrival or departure row stays null).
The stop-level flag `cancelled` (Step 1) is used by the summaries.

In [ ]:
for kind in ["arrival", "departure"]:
    flag = data[f"{kind}Cancelled"]
    data[f"{kind}Cancelled_resolved"] = np.where(flag.isna(), None, np.where(flag.fillna(False), "t", "f"))
print(f"{data['cancelled'].mean() * 100:.2f}% of stops cancelled (all countries)")

# Step 5 - Punctuality units and summaries

## 5a - Terminus punctuality (same metrics as the Chuuchuu `Summary` / `Summary_partial_alt` sheets)

One unit per journey, **on the German stops only**: `n_stops` / cancellations are counted over the German stops, the closing stop is the **last German stop** (the terminus for a domestic journey). A journey with a single German stop has no resolved closing stop (as in Chuuchuu).
Definitions are unchanged: delayed at X min = arrived (closing stop not cancelled) and arrival delay strictly above X minutes; a partially cancelled unit is never counted as delayed in the main measurement (`Summary`), and goes through the delay check if it reached its closing stop in the alternative one (`Summary_partial_alt`).

Column **N train arriving at terminus with measured delay**: the closing stop was reached and has an arrival delay. (The arrival row is missing for stops that are only a departure.)

In [ ]:
german = data[data["country"] == COUNTRY]          # sorted by jcode, stop_sequence
service_date = german["date"]
year = service_date.dt.year.to_numpy()
month = service_date.dt.month.to_numpy()

grouped = german.groupby("jcode")
first_rows = german.drop_duplicates("jcode").set_index("jcode")
journeys = first_rows[["journey_id", "routeType", "journey_type", "HSR_service", "LD", "night_train"]].copy()
journeys["year"] = first_rows["date"].dt.year
journeys["month"] = first_rows["date"].dt.month
journeys["operator"] = first_rows["operator_for_punctuality"].astype(object)
journeys["service_operator"] = first_rows["normalized_operator"].astype(object)
journeys["punctuality_scope"] = "German network"
journeys["stat_unit_id"] = journeys["journey_id"]

journeys["n_stops"] = grouped.size()
journeys["n_cancelled_stops"] = grouped["cancelled"].sum()
journeys["is_fully_cancelled"] = journeys["n_cancelled_stops"] == journeys["n_stops"]
journeys["is_partially_cancelled"] = (journeys["n_cancelled_stops"] > 0) & (journeys["n_cancelled_stops"] < journeys["n_stops"])

# closing stop = last German stop, resolved only when the unit has more than one German stop
closing_rows = german.drop_duplicates("jcode", keep="last").set_index("jcode")
resolved = journeys["n_stops"] > 1
journeys["cancelled_terminus"] = closing_rows["cancelled"].reindex(journeys.index).astype(object).where(resolved)          # NaN where unresolved
journeys["terminus_arrival_delay"] = closing_rows["arrivalDelay"].reindex(journeys.index).where(resolved)
journeys["arrived_at_terminus"] = journeys["cancelled_terminus"].eq(False)
journeys["arrived_at_terminus_measured"] = journeys["arrived_at_terminus"] & journeys["terminus_arrival_delay"].notna()

not_partially_cancelled = ~journeys["is_partially_cancelled"]
for threshold in DELAY_THRESHOLDS_MIN:
    journeys[f"delayed_{threshold}min"] = (
        journeys["arrived_at_terminus"] & not_partially_cancelled & (journeys["terminus_arrival_delay"] > threshold * 60)
    )
is_partial = journeys["is_partially_cancelled"]
journeys["is_partial_not_reaching_terminus"] = is_partial & ~journeys["arrived_at_terminus"]
journeys["is_partial_reaching_terminus"] = is_partial & journeys["arrived_at_terminus"]
for threshold in DELAY_THRESHOLDS_MIN:
    journeys[f"delayed_alt_{threshold}min"] = (
        journeys["arrived_at_terminus"] & (journeys["terminus_arrival_delay"] > threshold * 60)
    )
journeys = journeys.reset_index(drop=True)
journeys["operator"] = journeys["operator"].fillna("UNKNOWN_OPERATOR")

print(f"{len(journeys):,} punctuality units (journeys with at least one German stop)")
print(f"{journeys['cancelled_terminus'].isna().sum():,} units with no resolved closing stop")
print(f"{journeys['arrived_at_terminus'].sum():,} arrived at the closing stop, {journeys['arrived_at_terminus_measured'].sum():,} with a measured delay")
print(f"{journeys['is_fully_cancelled'].sum():,} fully / {journeys['is_partially_cancelled'].sum():,} partially cancelled")
print(", ".join(f"{journeys[f'delayed_{t}min'].sum():,} delayed >{t}min" for t in DELAY_THRESHOLDS_MIN))

In [ ]:
SUMMARY_KEYS = ["year", "month", "operator", "punctuality_scope", "routeType", "journey_type", "HSR_service", "LD", "night_train"]
SUMMARY_AGGREGATIONS = dict(
    n_total_train=("stat_unit_id", "size"),
    n_cancelled_full=("is_fully_cancelled", "sum"),
    n_cancelled_partial=("is_partially_cancelled", "sum"),
    **{f"n_delay_{t}min": (f"delayed_{t}min", "sum") for t in DELAY_THRESHOLDS_MIN},
    n_arriving_terminus=("arrived_at_terminus", "sum"),
    n_arriving_terminus_measured=("arrived_at_terminus_measured", "sum"),
    n_partial_reaching_terminus=("is_partial_reaching_terminus", "sum"),
    n_partial_not_reaching_terminus=("is_partial_not_reaching_terminus", "sum"),
    **{f"n_delay_alt_{t}min": (f"delayed_alt_{t}min", "sum") for t in DELAY_THRESHOLDS_MIN},
)

summary = journeys.groupby(SUMMARY_KEYS, dropna=False, observed=True).agg(**SUMMARY_AGGREGATIONS).reset_index()
summary["n_cancelled_full_or_partial"] = summary["n_cancelled_full"] + summary["n_cancelled_partial"]
summary["n_cancelled_full_or_partial_not_reaching"] = summary["n_cancelled_full"] + summary["n_partial_not_reaching_terminus"]
assert summary["n_total_train"].sum() == len(journeys), "grouped total doesn't match the unit count : a group key must be dropping nulls"

SUMMARY_COLUMN_NAMES = {
    "year": "Year", "month": "Month", "punctuality_scope": "Punctuality scope", "routeType": "Route type",
    "journey_type": "Journey type", "HSR_service": "HSR service", "LD": "Long distance", "night_train": "Night train",
    "n_total_train": "N total train",
    "n_cancelled_full": "N cancelled train (full cancellation)",
    "n_cancelled_partial": "N cancelled train (partial cancellation)",
    "n_cancelled_full_or_partial": "N cancelled train (full + partial)",
    **{f"n_delay_{t}min": f"N train delay (> {t} min)" for t in DELAY_THRESHOLDS_MIN},
    "n_arriving_terminus": "N train arriving at terminus",
    "n_arriving_terminus_measured": "N train arriving at terminus with measured delay",
    "n_partial_reaching_terminus": "N partially cancelled train reaching terminus",
    "n_partial_not_reaching_terminus": "N partially cancelled train not reaching terminus",
    "n_cancelled_full_or_partial_not_reaching": "N cancelled train (full + partial not reaching terminus)",
    **{f"n_delay_alt_{t}min": f"N train delay (> {t} min) [partial reaching terminus included]" for t in DELAY_THRESHOLDS_MIN},
}
SUMMARY_KEY_COLUMNS = ["Year", "Month", "operator", "Punctuality scope", "Route type", "Journey type", "HSR service", "Long distance", "Night train"]
SUMMARY_COUNT_COLUMNS = (
    ["N total train", "N cancelled train (full cancellation)", "N cancelled train (partial cancellation)", "N cancelled train (full + partial)"]
    + [f"N train delay (> {t} min)" for t in DELAY_THRESHOLDS_MIN]
    + ["N train arriving at terminus", "N train arriving at terminus with measured delay"]
)
SUMMARY_ALT_COUNT_COLUMNS = (
    ["N total train", "N cancelled train (full cancellation)", "N cancelled train (partial cancellation)",
     "N partially cancelled train reaching terminus", "N partially cancelled train not reaching terminus",
     "N cancelled train (full + partial not reaching terminus)"]
    + [f"N train delay (> {t} min) [partial reaching terminus included]" for t in DELAY_THRESHOLDS_MIN]
    + ["N train arriving at terminus", "N train arriving at terminus with measured delay"]
)


def format_summary(table, count_columns):
    table = table.rename(columns=SUMMARY_COLUMN_NAMES)
    return table[SUMMARY_KEY_COLUMNS + count_columns].sort_values(SUMMARY_KEY_COLUMNS)


summary_partial_alt = format_summary(summary, SUMMARY_ALT_COUNT_COLUMNS)
summary = format_summary(summary, SUMMARY_COUNT_COLUMNS)
summary

## 5b - Punctuality at all stops (sheet `Summary_all_stops`)

Every **German** stop of every long-distance train, not only the terminus:

- **Stop delay** = arrival delay at every stop that has a planned arrival; at a journey's first stop, which has none, the departure delay is used (same as the Swiss pipeline).
- **Delayed stop** = delay strictly above the threshold, for **3, 5, 6 and 15 minutes**. Each threshold's delayed stops are a subset of the lower one's.
- **A cancelled stop is a delayed stop** in the `Punctuality (X min, all stops)` columns: it counts as *not on time*.
- **Stops without a delay** (not cancelled, delay empty) are counted separately and left out of the denominator (`N stops not measured`).

`Punctuality (X min, all stops)` = 1 - (delayed at X min + cancelled stops) / (measured + cancelled stops), for X = 3, 5, 6 and 15.

**Added for the comparison with the official DB figure: 6 minutes, cancelled stops left out of the statistic** (neither delayed nor counted in the base):
- `Punctuality (> 6 min late, cancelled excluded)` = 1 - stops with delay **strictly above 6:00** / measured stops (the 6:00 exactly stop is on time, the rule of the other thresholds);
- `Punctuality (>= 6 min late, cancelled excluded)` = 1 - stops with delay **6:00 or more** / measured stops (a stop is on time only below 6 minutes). Delays are in whole minutes in this feed, so the two differ by the stops at exactly 6 minutes.

In [ ]:
stop_delay = german["arrivalDelay"].where(german["plannedArrival"].notna(), german["departureDelay"])
stop_cancelled = german["cancelled"]
stop_measured = stop_delay.notna() & ~stop_cancelled

stop_columns = {
    "Year": year, "Month": month,
    "operator": german["operator_for_punctuality"].astype(object).fillna("UNKNOWN_OPERATOR").to_numpy(),
    "Route type": german["routeType"].astype(object).to_numpy(),
    "Journey type": german["journey_type"].to_numpy(), "HSR service": german["HSR_service"].to_numpy(),
    "Long distance": german["LD"].to_numpy(), "Night train": german["night_train"].to_numpy(),
    "N stops": 1,
    "N cancelled stops": stop_cancelled.to_numpy(),
    "N stops with measured delay": stop_measured.to_numpy(),
    "N stops not measured": (~stop_measured & ~stop_cancelled).to_numpy(),
}
for t in ALL_STOPS_THRESHOLDS_MIN:
    stop_columns[f"N stops delayed (> {t} min)"] = (stop_measured & (stop_delay > t * 60)).to_numpy()
stop_columns["N stops delayed (>= 6 min)"] = (stop_measured & (stop_delay >= 6 * 60)).to_numpy()
stops = pd.DataFrame(stop_columns)

ALL_STOPS_KEYS = ["Year", "Month", "operator", "Route type", "Journey type", "HSR service", "Long distance", "Night train"]
summary_all_stops = stops.groupby(ALL_STOPS_KEYS, dropna=False).sum().reset_index()
del stops

judged = summary_all_stops["N stops with measured delay"] + summary_all_stops["N cancelled stops"]
for t in ALL_STOPS_THRESHOLDS_MIN:
    not_on_time = summary_all_stops[f"N stops delayed (> {t} min)"] + summary_all_stops["N cancelled stops"]
    summary_all_stops[f"N stops not on time (> {t} min or cancelled)"] = not_on_time
    summary_all_stops[f"Punctuality ({t} min, all stops)"] = 1 - not_on_time / judged.where(judged > 0)
measured = summary_all_stops["N stops with measured delay"]
summary_all_stops["Punctuality (> 6 min late, cancelled excluded)"] = 1 - summary_all_stops["N stops delayed (> 6 min)"] / measured.where(measured > 0)
summary_all_stops["Punctuality (>= 6 min late, cancelled excluded)"] = 1 - summary_all_stops["N stops delayed (>= 6 min)"] / measured.where(measured > 0)

count_columns = (["N stops", "N cancelled stops", "N stops with measured delay", "N stops not measured"]
                 + [f"N stops delayed (> {t} min)" for t in ALL_STOPS_THRESHOLDS_MIN] + ["N stops delayed (>= 6 min)"]
                 + [f"N stops not on time (> {t} min or cancelled)" for t in ALL_STOPS_THRESHOLDS_MIN]
                 + [f"Punctuality ({t} min, all stops)" for t in ALL_STOPS_THRESHOLDS_MIN]
                 + ["Punctuality (> 6 min late, cancelled excluded)", "Punctuality (>= 6 min late, cancelled excluded)"])
summary_all_stops = summary_all_stops[ALL_STOPS_KEYS + count_columns].sort_values(ALL_STOPS_KEYS)

tot = summary_all_stops[count_columns[:4] + [c for c in count_columns if c.startswith("N stops not on time") or c.startswith("N stops delayed")]].sum()
judged_total = tot["N stops with measured delay"] + tot["N cancelled stops"]
for t in ALL_STOPS_THRESHOLDS_MIN:
    print(f"all stops, {t} min : punctuality {(1 - tot[f'N stops not on time (> {t} min or cancelled)'] / judged_total) * 100:.2f}%")
print(f"all stops, 6 min, cancelled excluded : {(1 - tot['N stops delayed (> 6 min)'] / tot['N stops with measured delay']) * 100:.2f}% (> 6:00 late), "
      f"{(1 - tot['N stops delayed (>= 6 min)'] / tot['N stops with measured delay']) * 100:.2f}% (>= 6:00 late)")
print(f"{tot['N stops not measured'] / tot['N stops'] * 100:.2f}% of stops not measured")
summary_all_stops.head()

# Step 6 - Exports

In [ ]:
if export_summary.lower() == "y":
    os.makedirs("summary_stats", exist_ok=True)
    summary_path = f"summary_stats/german_summary_{data_selection}.xlsx"
    with pd.ExcelWriter(summary_path) as writer:
        summary.to_excel(writer, sheet_name="Summary", index=False)
        summary_partial_alt.to_excel(writer, sheet_name="Summary_partial_alt", index=False)
        summary_all_stops.to_excel(writer, sheet_name="Summary_all_stops", index=False)
        day_log.to_excel(writer, sheet_name="Day_log", index=False)
    print(f"Saved to {summary_path} (sheets: Summary, Summary_partial_alt, Summary_all_stops, Day_log)")

if export_enriched.lower() == "y":
    enriched_dir = DB_MOBILITHEK_DIR / "processed_data"
    enriched_dir.mkdir(exist_ok=True)
    enriched_path = enriched_dir / f"german_{data_selection}_enriched.parquet"
    data.drop(columns=["jcode"]).to_parquet(enriched_path)
    print(f"Saved to {enriched_path}")

## Summary of the process

In [ ]:
n_initial = row_counts["initial"]
n = row_counts["initial"] - row_counts["after_ld_filter"]
print(f"Dropped - not long distance / operational stops{n:>12,} rows ({n / n_initial * 100:6.2f}% of the final rows of the kept days)")
print(f"Excluded days (no real-time data): {len(excluded_days)} {excluded_days}")
print(f"Stop rows after merging arrival and departure: {row_counts['stop_rows']:,} (from {row_counts['after_ld_filter']:,} arrival / departure rows)")

# Step 7 - Sanity checks

In [ ]:
sanity_results = []


def sanity(check_name, ok, message, level="fail"):
    status = "OK" if ok else level.upper()
    sanity_results.append({"check": check_name, "status": status, "message": message})
    print(f"[{status}] {check_name}: {message}")


sanity("1a rows accounted", len(data) == row_counts["stop_rows"], f"{len(data)} rows, {row_counts['stop_rows']} expected")
sanity("1b journeys accounted", german["jcode"].nunique() == len(journeys), f"{len(journeys)} units, {german['jcode'].nunique()} journeys with a German stop")
sanity("1c summary total", summary["N total train"].sum() == len(journeys), f"summary sums to {summary['N total train'].sum()} trains, unit table has {len(journeys)}")
sanity("1d all-stops total", summary_all_stops["N stops"].sum() == len(german), f"{summary_all_stops['N stops'].sum()} stops in the sheet, {len(german)} German rows")

n_depart = (data["depart_terminus"] == "depart").groupby(data["jcode"]).sum()
n_terminus = (data["depart_terminus"] == "terminus").groupby(data["jcode"]).sum()
sanity("2a one depart per journey", (n_depart <= 1).all(), f"{(n_depart > 1).sum()} journeys with several 'depart' stops")
sanity("2b one terminus per journey", (n_terminus <= 1).all(), f"{(n_terminus > 1).sum()} journeys with several 'terminus' stops")
resolved_rows = data[data["depart_terminus"] != "unknown"]
bounds = resolved_rows.groupby("jcode")["stop_sequence"].agg(["min", "max"])
dep_s = resolved_rows.loc[resolved_rows["depart_terminus"] == "depart"].set_index("jcode")["stop_sequence"]
ter_s = resolved_rows.loc[resolved_rows["depart_terminus"] == "terminus"].set_index("jcode")["stop_sequence"]
sanity("2c depart is first, terminus is last", ((dep_s != bounds["min"].reindex(dep_s.index)).sum() + (ter_s != bounds["max"].reindex(ter_s.index)).sum()) == 0,
       "depart / terminus stop_sequence are the min / max of their journey")
planned_vs_seq = resolved_rows.dropna(subset=["sort_time"]).groupby("jcode")["sort_time"].apply(lambda s: s.is_monotonic_increasing)
sanity("2d stop_sequence follows planned time", planned_vs_seq.mean() >= 0.99, f"{(~planned_vs_seq).sum()} journeys where planned times are not increasing with stop_sequence", level="warn")

sanity("3a country known", ~data["country"].isin(["ERROR_INVALID_STOP_ID", "UNKNOWN_COUNTRY"]).any(), f"{data['country'].isin(['ERROR_INVALID_STOP_ID', 'UNKNOWN_COUNTRY']).sum()} stops with invalid / unknown country", level="warn")
sanity("3b operator mapped", ~data["normalized_operator"].str.startswith("OTHER ").any(), f"{data['normalized_operator'].str.startswith('OTHER ').sum()} stops with an unmapped operator code", level="warn")
sanity("3c one operator per journey", data.groupby("jcode")["normalized_operator"].nunique().max() == 1, "every journey has one operator")

both = journeys["is_fully_cancelled"] & journeys["is_partially_cancelled"]
sanity("4a full xor partial", both.sum() == 0, f"{both.sum()} units both fully and partially cancelled")
nested = (journeys["delayed_15min"] & ~journeys["delayed_5min"]) | (journeys["delayed_5min"] & ~journeys["arrived_at_terminus"])
sanity("4b delay flags nested", nested.sum() == 0, f"{nested.sum()} units break delayed_15min => delayed_5min => arrived")
sanity("4c partial split", (journeys["is_partial_reaching_terminus"].astype(int) + journeys["is_partial_not_reaching_terminus"].astype(int) == journeys["is_partially_cancelled"].astype(int)).all(),
       "reaching + not reaching = partially cancelled")
S = summary_all_stops
sanity("4d all-stops counts nested", ((S["N stops with measured delay"] + S["N cancelled stops"] + S["N stops not measured"]) == S["N stops"]).all(),
       "measured + cancelled + not measured = stops")
nested_stops = ((S["N stops delayed (> 15 min)"] > S["N stops delayed (> 6 min)"]) | (S["N stops delayed (> 6 min)"] > S["N stops delayed (> 5 min)"])
                | (S["N stops delayed (> 5 min)"] > S["N stops delayed (> 3 min)"]) | (S["N stops delayed (> 3 min)"] > S["N stops with measured delay"])
                | (S["N stops delayed (>= 6 min)"] < S["N stops delayed (> 6 min)"]) | (S["N stops delayed (>= 6 min)"] > S["N stops delayed (> 5 min)"]))
sanity("4e all-stops delay counts nested", nested_stops.sum() == 0, f"{nested_stops.sum()} rows break delayed_15 <= delayed_6 <= delayed_5 <= delayed_3 <= measured, delayed_6 <= delayed_>=6 <= delayed_5")

share_unmeasured = S["N stops not measured"].sum() / S["N stops"].sum()
sanity("5a measured share", share_unmeasured <= 0.10, f"{share_unmeasured * 100:.2f}% of German stops have no delay (ceiling 10%)", level="warn")
unmeasured_terminus = 1 - journeys["arrived_at_terminus_measured"].sum() / max(journeys["arrived_at_terminus"].sum(), 1)
sanity("5b terminus measured share", unmeasured_terminus <= 0.10, f"{unmeasured_terminus * 100:.2f}% of arrivals at the closing stop have no delay (ceiling 10%)", level="warn")
sanity("5c excluded days", len(excluded_days) == 0, f"{len(excluded_days)} day(s) without real-time data excluded: {excluded_days}", level="warn")
last_update = german["arrival_update"].fillna(german["departure_update"])
stop_time = german["arrival"].fillna(german["departure"])
early_snapshot = ((stop_time - last_update).dt.total_seconds() > 10 * 60).mean()
no_update = last_update.isna().mean()
sanity("5d final snapshot before the stop", early_snapshot <= 0.30,
       f"{early_snapshot * 100:.1f}% of German stops have a final snapshot more than 10 min before the stop time (taken as the actual delay, like the Swiss pipeline); {no_update * 100:.1f}% have no update timestamp", level="warn")

results = pd.DataFrame(sanity_results)
display(results)
assert not (results["status"] == "FAIL").any(), "at least one sanity check failed"